# Step 5: Outcome Analysis

**Study:** Synthetic RWE Diabetes Treatment Study  
**Comparison:** Metformin users (Group A) vs Other diabetes medication users (Group B)

**Methods:**
- Primary composite: diabetes complication, cardiovascular event, or hospitalization
- Event rates per 100 patient-years
- Kaplan-Meier survival analysis with log-rank test
- Unadjusted logistic regression (statsmodels)

**Important:** Results are unadjusted associations. Confounding should be addressed in subsequent causal inference analysis.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "synthea_csv"
REPORT_DIR = PROJECT_ROOT / "reports"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from outcome_analysis import run_outcome_analysis, TREATMENT_A, TREATMENT_B

## Outcome definitions

| Outcome | Source | Definition |
| --- | --- | --- |
| Primary composite | conditions + encounters | First post-index complication, CVD, or hospitalization/ED |
| Diabetes complication | conditions.csv | Nephropathy, neuropathy, retinopathy, T2DM sequelae |
| Cardiovascular event | conditions.csv | IHD, heart failure, stroke |
| Hospitalization | encounters.csv | Inpatient or emergency visit |
| Secondary | multiple | Encounter count, medication change, disease progression |

In [ ]:
result = run_outcome_analysis(DATA_DIR, REPORT_DIR)

outcome_events = result.outcome_events
followup = result.patient_followup
summary = result.outcome_summary
logistic = result.logistic_results
survival = result.survival_results

## Follow-up and outcome events

In [ ]:
print("Follow-up summary:")
display(
    followup.groupby("treatment_group")[["followup_days", "followup_months", "followup_years"]]
    .agg(["mean", "median"])
    .round(1)
)

print("\nSample outcome events:")
display(outcome_events.head(10))

## Event rates by treatment group

In [ ]:
primary = summary[
    summary["outcome_type"].isin([
        "primary_composite_event", "diabetes_complication",
        "cardiovascular_event", "hospitalization_event",
    ])
]
display(
    primary.pivot_table(
        index="outcome_type",
        columns="treatment_group",
        values=["n_events", "event_percentage", "events_per_100_patient_years"],
    )
)

## Statistical results

In [ ]:
print("Kaplan-Meier / log-rank (primary composite):")
display(survival)

print("\nLogistic regression (unadjusted, Other vs Metformin):")
display(logistic)

## Visualizations

In [ ]:
from IPython.display import Image, display

for fig_path in result.figure_paths:
    print(fig_path.name)
    display(Image(filename=str(fig_path)))

## Interpretation

These results represent **unadjusted associations** between treatment group and clinical outcomes. Other medication users show higher event rates for several outcomes, but this likely reflects confounding by indication (older, more comorbid patients initiated on insulin).

**Next steps:** Propensity score weighting and doubly robust estimation to adjust for baseline imbalances identified in Step 3.

Full report: `reports/outcome_analysis_report.html`